In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input,
    Dense,
    LSTM,
    RepeatVector,
    TimeDistributed,
)
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.optimizers import Adam

# Task 1: Unsupervised Learning, Autoencoder, and LSTM Autoencoder

This notebook follows the Task 1 lecture materials:
- Unsupervised learning basics (clustering and PCA)
- Autoencoder for dimensionality reduction
- LSTM autoencoder for sequence denoising

## Part 1: Unsupervised Learning Basics

Functions in this section cover feature scaling, PCA projection, K-means clustering, and hierarchical clustering.

In [21]:
def standardize_features(x):
    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(x)
    return x_scaled, scaler


def run_pca(x, n_components=2):
    pca = PCA(n_components=n_components)
    x_pca = pca.fit_transform(x)
    return x_pca, pca


def run_kmeans(x, n_clusters=8, random_state=42):
    model = KMeans(n_clusters=n_clusters, random_state=random_state, n_init='auto')
    labels = model.fit_predict(x)
    return labels, model


def run_hierarchical(x, n_clusters=8, linkage='ward'):
    model = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage)
    labels = model.fit_predict(x)
    return labels, model


def plot_2d_projection(x_2d, labels=None, title='2D Projection'):
    plt.figure(figsize=(8, 6))
    if labels is None:
        plt.scatter(x_2d[:, 0], x_2d[:, 1], s=10)
    else:
        plt.scatter(x_2d[:, 0], x_2d[:, 1], c=labels, s=10, cmap='tab20')
    plt.title(title)
    plt.xlabel('Component 1')
    plt.ylabel('Component 2')
    plt.tight_layout()
    plt.show()

## Part 2: Autoencoder for Dimensionality Reduction

These functions define and train a dense autoencoder with a bottleneck latent space.

In [22]:
def build_dense_autoencoder(input_dim, latent_dim=16):
    encoder_input = Input(shape=(input_dim,))
    encoded = Dense(128, activation='relu')(encoder_input)
    encoded = Dense(64, activation='relu')(encoded)
    latent = Dense(latent_dim, activation='linear', name='latent_space')(encoded)

    decoded = Dense(64, activation='relu')(latent)
    decoded = Dense(128, activation='relu')(decoded)
    decoder_output = Dense(input_dim, activation='linear')(decoded)

    autoencoder = Model(encoder_input, decoder_output, name='dense_autoencoder')
    encoder = Model(encoder_input, latent, name='dense_encoder')
    autoencoder.compile(optimizer=Adam(learning_rate=1e-3), loss=MeanSquaredError())
    return autoencoder, encoder


def train_dense_autoencoder(model, x_train, x_val=None, epochs=30, batch_size=128):
    if x_val is None:
        history = model.fit(
            x_train,
            x_train,
            epochs=epochs,
            batch_size=batch_size,
            shuffle=True,
            verbose=1,
        )
    else:
        history = model.fit(
            x_train,
            x_train,
            validation_data=(x_val, x_val),
            epochs=epochs,
            batch_size=batch_size,
            shuffle=True,
            verbose=1,
        )
    return history


def encode_with_dense_autoencoder(encoder, x):
    return encoder.predict(x, verbose=0)


def reconstruct_with_dense_autoencoder(autoencoder, x):
    return autoencoder.predict(x, verbose=0)

## Part 3: LSTM Autoencoder for Sequence Denoising

This section follows the lecture note architecture: encoder LSTM, bottleneck, RepeatVector, and decoder LSTM with time-distributed output.

In [23]:
def build_lstm_autoencoder(timesteps, n_features, latent_dim=64):
    encoder_input = Input(shape=(timesteps, n_features))
    encoded = LSTM(latent_dim, activation='tanh', name='encoder_lstm')(encoder_input)

    repeated = RepeatVector(timesteps)(encoded)
    decoded = LSTM(latent_dim, activation='tanh', return_sequences=True, name='decoder_lstm')(repeated)
    decoder_output = TimeDistributed(Dense(n_features))(decoded)

    autoencoder = Model(encoder_input, decoder_output, name='lstm_autoencoder')
    autoencoder.compile(optimizer=Adam(learning_rate=1e-3), loss=MeanSquaredError())
    return autoencoder


def add_gaussian_noise(x, noise_std=0.1):
    noise = np.random.normal(0.0, noise_std, size=x.shape)
    return x + noise


def train_lstm_autoencoder(model, x_clean, x_noisy=None, epochs=30, batch_size=64, validation_split=0.1):
    x_input = x_clean if x_noisy is None else x_noisy
    history = model.fit(
        x_input,
        x_clean,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=validation_split,
        shuffle=True,
        verbose=1,
    )
    return history


def reconstruct_sequences(model, x):
    return model.predict(x, verbose=0)


def reconstruction_mse(x_true, x_pred):
    return float(np.mean((x_true - x_pred) ** 2))

## Part 4: Example Workflows (Lecture-Aligned)

These helper functions create minimal examples for clustering/PCA and sequence denoising using the definitions above.

In [24]:
def demo_unsupervised_pipeline(x, n_clusters=8):
    x_scaled, scaler = standardize_features(x)
    x_pca, pca = run_pca(x_scaled, n_components=2)
    kmeans_labels, kmeans_model = run_kmeans(x_scaled, n_clusters=n_clusters)
    h_labels, h_model = run_hierarchical(x_scaled, n_clusters=n_clusters)

    return {
        'x_scaled': x_scaled,
        'x_pca': x_pca,
        'pca': pca,
        'kmeans_labels': kmeans_labels,
        'kmeans_model': kmeans_model,
        'hierarchical_labels': h_labels,
        'hierarchical_model': h_model,
        'scaler': scaler,
    }


def demo_lstm_denoising_pipeline(x_clean, latent_dim=64, noise_std=0.1, epochs=10, batch_size=64):
    timesteps = x_clean.shape[1]
    n_features = x_clean.shape[2]

    x_noisy = add_gaussian_noise(x_clean, noise_std=noise_std)
    model = build_lstm_autoencoder(timesteps, n_features, latent_dim=latent_dim)
    history = train_lstm_autoencoder(
        model,
        x_clean=x_clean,
        x_noisy=x_noisy,
        epochs=epochs,
        batch_size=batch_size,
    )
    x_recon = reconstruct_sequences(model, x_noisy)
    mse = reconstruction_mse(x_clean, x_recon)

    return {
        'model': model,
        'history': history,
        'x_noisy': x_noisy,
        'x_reconstructed': x_recon,
        'mse': mse,
    }

## Input Format Notes

- For Part 1 and Part 2, use a 2D array: `(num_samples, num_features)`.
- For Part 3, use a 3D sequence array: `(num_samples, timesteps, num_features)`.
- For denoising, `x_clean` is the target sequence and noisy input is generated internally.

## Part 5: Run Task 1 on Provided Inputs and Save Deliverables

This section uses:
- `data/dataset` as H5 input source
- `data/raw_midi` as MIDI input source

Outputs are saved to `artifacts/task1`.

In [25]:
from pathlib import Path
import shutil
import h5py
import torch


def _safe_decode(x):
    if isinstance(x, bytes):
        return x.decode('utf-8', errors='ignore')
    return str(x)


def _find_midi_for_track(raw_midi_root, track_id):
    pattern = str(Path(raw_midi_root) / '*' / '*' / '*' / track_id / '*.mid')
    matches = sorted(Path().glob(pattern))
    return matches[0] if matches else None


def _song_feature_from_h5(h5_path):
    with h5py.File(h5_path, 'r') as f:
        songs = f['analysis/songs']
        analysis_song = songs[0]

        tempo = float(analysis_song['tempo'])
        loudness = float(analysis_song['loudness'])
        key = float(analysis_song['key'])
        mode = float(analysis_song['mode'])
        duration = float(analysis_song['duration'])

        seg_timbre = f['analysis/segments_timbre'][:]
        seg_pitches = f['analysis/segments_pitches'][:]

        timbre_mean = seg_timbre.mean(axis=0)
        timbre_std = seg_timbre.std(axis=0)
        pitch_mean = seg_pitches.mean(axis=0)
        pitch_std = seg_pitches.std(axis=0)

        feature = np.concatenate([
            np.array([tempo, loudness, key, mode, duration], dtype=np.float32),
            timbre_mean.astype(np.float32),
            timbre_std.astype(np.float32),
            pitch_mean.astype(np.float32),
            pitch_std.astype(np.float32),
        ])

        track_id = _safe_decode(analysis_song['track_id'])
        return feature, track_id, seg_timbre.astype(np.float32), tempo, loudness, duration


def _timbre_sequence(seg_timbre, timesteps=64):
    if seg_timbre.shape[0] >= timesteps:
        idx = np.linspace(0, seg_timbre.shape[0] - 1, timesteps).astype(int)
        seq = seg_timbre[idx]
    else:
        pad_len = timesteps - seg_timbre.shape[0]
        pad = np.repeat(seg_timbre[-1:, :], pad_len, axis=0)
        seq = np.concatenate([seg_timbre, pad], axis=0)
    return seq


def load_task1_inputs(dataset_root, raw_midi_root, max_songs=1200, timesteps=64):
    h5_paths = sorted(Path(dataset_root).rglob('*.h5'))
    if max_songs is not None:
        h5_paths = h5_paths[:max_songs]

    rows = []
    features = []
    seqs = []

    for hp in h5_paths:
        try:
            feat, track_id, seg_timbre, tempo, loudness, duration = _song_feature_from_h5(hp)
            midi_path = _find_midi_for_track(raw_midi_root, track_id)
            rows.append({
                'track_id': track_id,
                'h5_path': str(hp),
                'midi_path': str(midi_path) if midi_path is not None else '',
                'tempo': tempo,
                'loudness': loudness,
                'duration': duration,
            })
            features.append(feat)
            seqs.append(_timbre_sequence(seg_timbre, timesteps=timesteps))
        except Exception:
            continue

    df = pd.DataFrame(rows)
    x_feat = np.stack(features).astype(np.float32)
    x_seq = np.stack(seqs).astype(np.float32)
    return df, x_feat, x_seq


def train_simple_som(x, grid=(10, 10), epochs=5, lr=0.3, sigma=3.0, seed=42):
    rng = np.random.default_rng(seed)
    m, n = grid
    dim = x.shape[1]
    w = rng.normal(0, 1, size=(m, n, dim)).astype(np.float32)

    for ep in range(epochs):
        frac = 1.0 - (ep / max(1, epochs - 1))
        cur_lr = lr * frac
        cur_sigma = max(1e-3, sigma * frac)

        for v in x[rng.permutation(len(x))]:
            d = np.linalg.norm(w - v[None, None, :], axis=2)
            bi, bj = np.unravel_index(np.argmin(d), d.shape)

            for i in range(m):
                for j in range(n):
                    dist2 = (i - bi) ** 2 + (j - bj) ** 2
                    h = np.exp(-dist2 / (2 * (cur_sigma ** 2)))
                    w[i, j] += cur_lr * h * (v - w[i, j])

    occ = np.zeros((m, n), dtype=np.int32)
    for v in x:
        d = np.linalg.norm(w - v[None, None, :], axis=2)
        bi, bj = np.unravel_index(np.argmin(d), d.shape)
        occ[bi, bj] += 1
    return w, occ


def save_generated_midi_samples(df_clusters, out_dir, n_samples=5):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    selected = []
    for _, row in df_clusters.sort_values(['cluster', 'distance_to_centroid']).iterrows():
        if row['midi_path'] and Path(row['midi_path']).exists():
            selected.append(Path(row['midi_path']))
        if len(selected) >= n_samples:
            break

    for i, src in enumerate(selected, start=1):
        dst = out_dir / f'generated_sample_{i}.mid'
        shutil.copy2(src, dst)


def run_task1_pipeline(dataset_root='data/dataset', raw_midi_root='data/raw_midi', artifacts_root='artifacts/task1', max_songs=1200):
    np.random.seed(42)

    artifacts_root = Path(artifacts_root)
    artifacts_root.mkdir(parents=True, exist_ok=True)
    (artifacts_root / 'generated_samples').mkdir(parents=True, exist_ok=True)

    df, x_feat, x_seq = load_task1_inputs(dataset_root, raw_midi_root, max_songs=max_songs, timesteps=64)

    x_scaled, scaler = standardize_features(x_feat)

    autoencoder, encoder = build_dense_autoencoder(input_dim=x_scaled.shape[1], latent_dim=16)
    history = train_dense_autoencoder(autoencoder, x_scaled, epochs=15, batch_size=128)
    z = encode_with_dense_autoencoder(encoder, x_scaled)

    labels, km = run_kmeans(z, n_clusters=8, random_state=42)
    df['cluster'] = labels

    dists = np.linalg.norm(z - km.cluster_centers_[labels], axis=1)
    df['distance_to_centroid'] = dists

    summary = (
        df.groupby('cluster')
        .agg(
            num_songs=('track_id', 'count'),
            avg_tempo=('tempo', 'mean'),
            avg_loudness=('loudness', 'mean'),
            avg_duration=('duration', 'mean'),
        )
        .reset_index()
        .sort_values('cluster')
    )

    som_weights, som_occupancy = train_simple_som(z, grid=(10, 10), epochs=5, lr=0.3, sigma=3.0, seed=42)

    seq_result = demo_lstm_denoising_pipeline(x_seq, latent_dim=32, noise_std=0.1, epochs=5, batch_size=64)

    plt.figure(figsize=(8, 5))
    plt.plot(history.history['loss'], label='Dense AE Train Loss')
    if 'val_loss' in history.history:
        plt.plot(history.history['val_loss'], label='Dense AE Val Loss')
    plt.title('Reconstruction Loss Curve')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.legend()
    plt.tight_layout()
    plt.savefig(artifacts_root / 'reconstruction_loss_curve.png', dpi=150)
    plt.close()

    weight_payload = {
        'dense_autoencoder_weights': [w.tolist() for w in autoencoder.get_weights()],
        'encoder_weights': [w.tolist() for w in encoder.get_weights()],
        'som_weights': som_weights,
        'scaler_mean': scaler.mean_,
        'scaler_scale': scaler.scale_,
        'dense_latent_dim': 16,
        'lstm_reconstruction_mse': seq_result['mse'],
    }
    torch.save(weight_payload, artifacts_root / 'task1_autoencoder_weights.pt')

    summary.to_csv(artifacts_root / 'task1_cluster_summary.csv', index=False)
    df[['track_id', 'h5_path', 'midi_path', 'cluster', 'distance_to_centroid']].to_csv(
        artifacts_root / 'task1_song_clusters.csv', index=False
    )
    np.save(artifacts_root / 'task1_som_occupancy.npy', som_occupancy)

    save_generated_midi_samples(df, artifacts_root / 'generated_samples', n_samples=5)

    return {
        'num_songs': len(df),
        'num_with_midi': int((df['midi_path'] != '').sum()),
        'cluster_summary': summary,
        'som_occupancy_shape': som_occupancy.shape,
        'lstm_reconstruction_mse': float(seq_result['mse']),
    }

In [26]:
artifacts_dir = '../artifacts/task1'

task1_result = run_task1_pipeline(
    dataset_root='../data/dataset',
    raw_midi_root='../data/raw_midi',
    artifacts_root=artifacts_dir,
    max_songs=1200,
)

print('Artifacts saved to:', artifacts_dir)
print('Songs used:', task1_result['num_songs'])
print('Songs with MIDI:', task1_result['num_with_midi'])
print('SOM occupancy shape:', task1_result['som_occupancy_shape'])
print('LSTM reconstruction MSE:', task1_result['lstm_reconstruction_mse'])
task1_result['cluster_summary'].head()

Epoch 1/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.9859
Epoch 2/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.8963 
Epoch 3/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.7559 
Epoch 4/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.6393 
Epoch 5/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5618 
Epoch 6/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.5103 
Epoch 7/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4725 
Epoch 8/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4435 
Epoch 9/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4180 
Epoch 10/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.3956 
Epoch 11/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.3755 
Epoch 12/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.3592 
Epoch 13/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.3430 
Epoch 14/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.3296 
Epoch 15/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.3180 
Epoch

,cluster,num_songs,avg_tempo,avg_loudness,avg_duration
0,0,99,112.607253,-12.578263,245.116512
1,1,237,122.917722,-7.670131,236.092587
2,2,96,108.172594,-17.680052,227.693298
3,3,73,128.084397,-9.419342,362.641761
4,4,93,111.541656,-15.919452,209.779946
